# Text to Numbers - Learn Step by Step

Run each cell one by one and observe the output!

---
# WHY do we need this?

Machine Learning models only understand **numbers**, not text.

```
"I love AI"  →  ML Model  →  ❌ Error!
[0.2, 0.8]   →  ML Model  →  ✅ Works!
```

So we need to convert: **TEXT → NUMBERS**

---
---
# PART 1: ONE-HOT ENCODING

**WHAT:** Each word gets a unique position with "1", rest are "0"

**WHY:** Simplest way to represent words as numbers

## Step 1: Start with a sentence

In [ ]:
# Our sentence
sentence = "I love AI"

print("Our sentence:", sentence)
print("\nProblem: ML model cannot read this text!")
print("Solution: Convert to numbers")

## Step 2: Tokenization (Split into words)

**WHAT:** Break sentence into individual words

**WHY:** We encode WORDS, not entire sentences

In [ ]:
# Split sentence into words
# .lower() = convert to lowercase ("AI" → "ai")
# .split() = split by spaces

words = sentence.lower().split()

print("Original:", sentence)
print("After lowercase:", sentence.lower())
print("After split:", words)
print("\nWe now have", len(words), "words to encode")

## Step 3: Create Vocabulary

**WHAT:** List of all unique words (sorted alphabetically)

**WHY:** Each word needs a fixed position in the vector

In [ ]:
# Get unique words and sort them
vocabulary = sorted(set(words))

print("Words:", words)
print("Vocabulary (unique + sorted):", vocabulary)
print("\nEach word gets a position:")
for position, word in enumerate(vocabulary):
    print(f"  Position {position}: '{word}'")

## Step 4: One-Hot Encode (Manual Way)

**WHAT:** Create vector where only the word's position is "1"

**WHY:** Now each word is a unique number pattern!

In [ ]:
# Manual one-hot encoding
print("Vocabulary:", vocabulary)
print("Positions:  ", list(range(len(vocabulary))))
print("\nOne-Hot Encoding:")
print("-" * 40)

for word in words:
    # Create vector of zeros
    vector = [0] * len(vocabulary)
    
    # Find word's position and set to 1
    position = vocabulary.index(word)
    vector[position] = 1
    
    print(f"'{word}' → position {position} → {vector}")

## Step 5: Using sklearn (The Real Way)

**WHY sklearn?** 
- Handles everything automatically
- Works with large data
- Industry standard

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# sklearn needs data in format: [[word1], [word2], [word3]]
words_for_sklearn = [[word] for word in words]

print("Our words:", words)
print("Format sklearn needs:", words_for_sklearn)

## Step 6: fit() - Learn the Vocabulary

**WHAT:** Encoder looks at data and learns all unique words

**WHY:** Must know vocabulary BEFORE converting

**Analogy:** Teacher learning student names before taking attendance

In [ ]:
# Create encoder
encoder = OneHotEncoder(sparse_output=False)

# FIT = Learn vocabulary
encoder.fit(words_for_sklearn)

print("✅ Encoder has learned the vocabulary!")
print("\nVocabulary learned:", encoder.categories_[0])
print("Number of words:", len(encoder.categories_[0]))

## Step 7: transform() - Convert to Vectors

**WHAT:** Use learned vocabulary to convert words to numbers

**WHY:** Now we can actually get the one-hot vectors

**Analogy:** Teacher taking attendance using the names they learned

In [ ]:
# TRANSFORM = Convert using learned vocabulary
vectors = encoder.transform(words_for_sklearn)

print("Vocabulary:", list(encoder.categories_[0]))
print("\nOne-Hot Vectors:")
print("-" * 40)

for word, vector in zip(words, vectors):
    print(f"'{word}' → {vector.astype(int)}")

## ⚠️ Common Error: Not Fitted!

**WHAT HAPPENS:** If you try transform() before fit()

**WHY ERROR:** Encoder doesn't know the vocabulary yet!

In [ ]:
# Create NEW encoder (not fitted)
new_encoder = OneHotEncoder(sparse_output=False)

# Try to transform WITHOUT fit
try:
    new_encoder.transform(words_for_sklearn)
except Exception as e:
    print("❌ ERROR:", type(e).__name__)
    print("Message:", e)
    print("\n💡 Solution: Call fit() first, then transform()")

## Step 8: fit_transform() - Shortcut!

**WHAT:** fit() + transform() in one step

**WHY:** More convenient, same result

In [ ]:
# Create encoder and do both steps at once
encoder2 = OneHotEncoder(sparse_output=False)
vectors2 = encoder2.fit_transform(words_for_sklearn)

print("fit_transform() = fit() + transform()")
print("\nResult:")
for word, vector in zip(words, vectors2):
    print(f"'{word}' → {vector.astype(int)}")

## ❓ Problem with One-Hot Encoding

What if a word appears multiple times?

In [ ]:
# Sentence with repeated word
sentence2 = "I love love love AI"
words2 = sentence2.lower().split()

print("Sentence:", sentence2)
print("Words:", words2)

# One-hot encode
words2_sklearn = [[w] for w in words2]
encoder3 = OneHotEncoder(sparse_output=False)
vectors3 = encoder3.fit_transform(words2_sklearn)

print("\nOne-Hot Vectors:")
for word, vector in zip(words2, vectors3):
    print(f"'{word}' → {vector.astype(int)}")

print("\n⚠️ Problem: 'love' appears 3 times but vectors are IDENTICAL!")
print("   One-Hot doesn't capture FREQUENCY!")
print("\n💡 Solution: Use Bag of Words (BOW)")

---
---
# PART 2: BAG OF WORDS (BOW)

**WHAT:** Count how many times each word appears

**WHY:** Captures word frequency (how often words appear)

## Step 1: Manual BOW

In [ ]:
sentence = "I love love love AI"
words = sentence.lower().split()

print("Sentence:", sentence)
print("Words:", words)

# Count manually
from collections import Counter
word_counts = Counter(words)

print("\nWord Counts:")
for word, count in sorted(word_counts.items()):
    print(f"  '{word}': {count} time(s)")

In [ ]:
# Create BOW vector manually
vocabulary = sorted(set(words))
bow_vector = [word_counts[word] for word in vocabulary]

print("Vocabulary:", vocabulary)
print("BOW Vector:", bow_vector)
print("\nMeaning:")
for word, count in zip(vocabulary, bow_vector):
    print(f"  '{word}' appears {count} time(s)")

## Step 2: BOW with sklearn

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# BOW works on DOCUMENTS (full sentences), not individual words
documents = [
    "I love love love AI"
]

# Create and fit_transform
bow = CountVectorizer()
bow_matrix = bow.fit_transform(documents)

print("Document:", documents[0])
print("\nVocabulary:", list(bow.get_feature_names_out()))
print("BOW Vector:", bow_matrix.toarray()[0])

## Step 3: BOW with Multiple Documents

**WHY multiple documents?** 
- Real-world has many documents
- Vocabulary is built from ALL documents
- Each document gets its own vector

In [ ]:
documents = [
    "people watch movie",
    "people watch cricket",
    "people like movie",
    "people like cricket"
]

print("Documents:")
for i, doc in enumerate(documents):
    print(f"  Doc {i+1}: {doc}")

In [ ]:
# Fit BOW on all documents
bow = CountVectorizer()
bow_matrix = bow.fit_transform(documents)

vocabulary = bow.get_feature_names_out()
print("Vocabulary (from ALL documents):", list(vocabulary))
print("\nMatrix shape:", bow_matrix.shape)
print("  → 4 documents, 5 unique words")

In [ ]:
# Show BOW for each document
print("Vocabulary:", list(vocabulary))
print("=" * 50)

for i, (doc, vector) in enumerate(zip(documents, bow_matrix.toarray())):
    print(f"\nDoc {i+1}: '{doc}'")
    print(f"Vector: {vector}")
    print("Meaning:", end=" ")
    for word, count in zip(vocabulary, vector):
        if count > 0:
            print(f"{word}={count}", end="  ")
    print()

## ❓ Problem with BOW

Common words get high counts but aren't meaningful!

In [ ]:
# Notice: "people" appears in ALL documents
print("Word 'people' count in each document:")
people_idx = list(vocabulary).index('people')

for i, vector in enumerate(bow_matrix.toarray()):
    print(f"  Doc {i+1}: {vector[people_idx]}")

print("\n⚠️ Problem: 'people' appears everywhere!")
print("   It's NOT useful for distinguishing documents.")
print("\n💡 Solution: Use TF-IDF (reduce common word importance)")

---
---
# PART 3: TF-IDF

**WHAT:** Weight words by importance
- Common words (appear everywhere) → LOW score
- Rare words (appear in few docs) → HIGH score

**WHY:** Find words that actually MATTER

## Understanding TF-IDF Formula

```
TF  = Term Frequency     = How often in THIS document?
IDF = Inverse Doc Freq   = How RARE across ALL documents?

TF-IDF = TF × IDF
```

**Example:**
- "people" in all 4 docs → IDF is LOW → TF-IDF is LOW
- "movie" in 2 docs → IDF is MEDIUM → TF-IDF is MEDIUM
- "cricket" in 2 docs → IDF is MEDIUM → TF-IDF is MEDIUM

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "people watch movie",
    "people watch cricket",
    "people like movie",
    "people like cricket"
]

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(documents)

vocabulary = tfidf.get_feature_names_out()
print("Vocabulary:", list(vocabulary))

In [ ]:
# Show TF-IDF for each document
import numpy as np

print("Vocabulary:", list(vocabulary))
print("=" * 60)

for i, (doc, vector) in enumerate(zip(documents, tfidf_matrix.toarray())):
    print(f"\nDoc {i+1}: '{doc}'")
    print(f"Vector: {np.round(vector, 2)}")

## Compare: BOW vs TF-IDF

In [ ]:
# Compare "people" (common) vs "movie" (less common)
print("Comparing word importance:\n")

people_idx = list(vocabulary).index('people')
movie_idx = list(vocabulary).index('movie')

print("Word: 'people' (appears in ALL 4 documents)")
print("-" * 40)
for i, (bow_vec, tfidf_vec) in enumerate(zip(bow_matrix.toarray(), tfidf_matrix.toarray())):
    print(f"  Doc {i+1}: BOW={bow_vec[people_idx]}  TF-IDF={tfidf_vec[people_idx]:.2f}")

print("\nWord: 'movie' (appears in 2 documents)")
print("-" * 40)
for i, (bow_vec, tfidf_vec) in enumerate(zip(bow_matrix.toarray(), tfidf_matrix.toarray())):
    print(f"  Doc {i+1}: BOW={bow_vec[movie_idx]}  TF-IDF={tfidf_vec[movie_idx]:.2f}")

print("\n✅ Notice: 'people' has LOWER TF-IDF than 'movie'!")
print("   TF-IDF correctly identifies 'movie' as more important.")

## Find Most Important Words

In [ ]:
print("Most important words in each document:\n")

for i, (doc, vector) in enumerate(zip(documents, tfidf_matrix.toarray())):
    # Sort by TF-IDF score (highest first)
    sorted_indices = np.argsort(vector)[::-1]
    
    print(f"Doc {i+1}: '{doc}'")
    print("  Important words:", end=" ")
    
    for idx in sorted_indices[:2]:  # Top 2
        if vector[idx] > 0:
            print(f"{vocabulary[idx]}({vector[idx]:.2f})", end="  ")
    print("\n")

---
---
# PART 4: THE COMPLETE WORKFLOW

In real ML projects, you have:
1. **Training data** - fit() + transform()
2. **Test data** - only transform() (use SAME vocabulary!)

In [ ]:
# TRAINING DATA
train_docs = [
    "I love machine learning",
    "deep learning is great",
    "python is awesome for AI"
]

# TEST DATA (new, unseen documents)
test_docs = [
    "I love python",
    "machine learning is great"
]

print("Training documents:")
for doc in train_docs:
    print(f"  - {doc}")

print("\nTest documents:")
for doc in test_docs:
    print(f"  - {doc}")

In [ ]:
# Step 1: Create vectorizer
vectorizer = TfidfVectorizer()

# Step 2: FIT on training data (learn vocabulary)
vectorizer.fit(train_docs)

print("✅ Vocabulary learned from training data:")
print(list(vectorizer.get_feature_names_out()))

In [ ]:
# Step 3: TRANSFORM training data
train_vectors = vectorizer.transform(train_docs)

print("Training vectors shape:", train_vectors.shape)
print("(3 documents, 9 words in vocabulary)")

In [ ]:
# Step 4: TRANSFORM test data (using SAME vocabulary!)
test_vectors = vectorizer.transform(test_docs)

print("Test vectors shape:", test_vectors.shape)
print("(2 documents, 9 words - same vocabulary!)")

print("\n⚠️ Important: We used transform(), NOT fit_transform()!")
print("   Test data must use the SAME vocabulary as training.")

In [ ]:
# What about unknown words?
new_doc = ["quantum computing is revolutionary"]
new_vector = vectorizer.transform(new_doc)

print("New document:", new_doc[0])
print("Vector:", new_vector.toarray()[0])

# Check which words were recognized
vocab = vectorizer.get_feature_names_out()
print("\nWords recognized:")
for word, score in zip(vocab, new_vector.toarray()[0]):
    if score > 0:
        print(f"  '{word}': {score:.2f}")

print("\n⚠️ 'quantum', 'computing', 'revolutionary' are NOT in vocabulary!")
print("   Only 'is' was recognized.")

---
---
# SUMMARY

## Three Techniques:

| Technique | What it does | When to use |
|-----------|--------------|-------------|
| **One-Hot** | Binary: 1 for word, 0 for others | Simple tasks, embeddings input |
| **BOW** | Count word frequency | When frequency matters |
| **TF-IDF** | Weight by importance | Document search, classification |

## Key Functions:

| Function | What it does | When to use |
|----------|--------------|-------------|
| `fit()` | Learn vocabulary | On training data |
| `transform()` | Convert to vectors | On any data |
| `fit_transform()` | Both at once | On training data (shortcut) |

## Common Errors:

1. **NotFittedError** → Call `fit()` before `transform()`
2. **Different vocabulary** → Don't `fit()` on test data!

In [ ]:
print("🎉 Congratulations! You've learned:")
print("   ✅ One-Hot Encoding")
print("   ✅ Bag of Words (BOW)")
print("   ✅ TF-IDF")
print("   ✅ fit() vs transform() vs fit_transform()")
print("\nYou're ready to use text vectorization in ML projects!")